# GemmaFit v5.1 E2B Multimodal Merge + LiteRT-LM Export

Experimental Colab path:

```text
google/gemma-4-E2B-it
+ GemmaFit v5.1 LoRA adapter
-> AutoModelForImageTextToText / AutoProcessor merge
-> LiteRT-LM export with vision/audio-capable graph when the exporter supports it
-> Pixel smoke through content://com.gemmafit.debug/litert_image_prompt_infer
```

This notebook does not retrain v5.1 and does not replace the existing text-router notebook. It writes Drive state, phase markers, event logs, disconnect points, and a done marker so interrupted Colab sessions can resume safely.

Android-ready means the Pixel image smoke route passes. If Pixel reports `TF_LITE_VISION_ENCODER not found in the model`, the merge may be correct but the exporter emitted a text-only LiteRT-LM graph.


In [ ]:
# Phase -1 / 0: Drive, repo, paths, durable state helpers.
import hashlib, json, os, shutil, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    userdata = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

REPO_URL = os.environ.get('GEMMAFIT_REPO_URL', 'https://github.com/bkttt0429/GemmaFit.git')
REPO_DIR = Path(os.environ.get('GEMMAFIT_REPO_DIR', '/content/GemmaFit'))
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
elif os.environ.get('SKIP_GIT_PULL', '0') != '1':
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--prune'], check=False)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
os.chdir(REPO_DIR)

def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    if userdata is not None:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return None

HF_TOKEN = secret('HF_TOKEN') or secret('HUGGINGFACE_TOKEN')
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

DRIVE_ROOT = Path(os.environ.get('GEMMAFIT_DRIVE_ROOT', '/content/drive/MyDrive/GemmaFit_train'))
TRAINED = DRIVE_ROOT / 'trained_outputs'
BASE_MODEL_ID = os.environ.get('BASE_MODEL_ID', 'google/gemma-4-E2B-it')
SOURCE_RUN_NAME = os.environ.get('SOURCE_RUN_NAME', 'google_gemma_4_E2B_it_gemmafit_v5_1_e2b_evidence_router_hard_cases')
ADAPTER_DIR = Path(os.environ.get('ADAPTER_DIR', str(TRAINED / 'adapter' / SOURCE_RUN_NAME)))
MULTIMODAL_RUN_SUFFIX = os.environ.get('MULTIMODAL_RUN_SUFFIX', 'gemmafit_v5_1_e2b_multimodal_router')
MULTIMODAL_RUN_NAME = os.environ.get('MULTIMODAL_RUN_NAME', 'google_gemma_4_E2B_it_gemmafit_v5_1_e2b_multimodal_router')
MERGED_MM_DIR = Path(os.environ.get('MERGED_MM_DIR', str(TRAINED / 'merged_multimodal_fp16' / MULTIMODAL_RUN_NAME)))
EXPORT_ROOT = Path(os.environ.get('EXPORT_ROOT', str(TRAINED / 'litert_export_v5_e2b_multimodal')))
LOCAL_WORK_ROOT = Path(os.environ.get('LOCAL_WORK_ROOT', '/content/gemmafit_v5_e2b_multimodal_work'))
ARTIFACT_PREFIX = os.environ.get('ARTIFACT_PREFIX', 'gemmafit-v5-e2b-evidence-router-mm')
LITERTLM_FINAL = EXPORT_ROOT / f'{ARTIFACT_PREFIX}.litertlm'
BUNDLE_ZIP = EXPORT_ROOT / f'{ARTIFACT_PREFIX}-local-artifacts.zip'
EXPORT_VARIANTS = [
    {'name': 'mm_wi8_ctx4096_prefill1024', 'cache_length': 4096, 'prefill_lengths': '[1024]'},
    {'name': 'mm_wi8_ctx2048_prefill1024', 'cache_length': 2048, 'prefill_lengths': '[1024]'},
    {'name': 'mm_wi8_ctx2048_prefill512', 'cache_length': 2048, 'prefill_lengths': '[512]'},
]

for path in [DRIVE_ROOT, TRAINED, MERGED_MM_DIR.parent, EXPORT_ROOT, LOCAL_WORK_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

RUN_STATE = EXPORT_ROOT / f'RUN_STATE_{MULTIMODAL_RUN_NAME}.json'
RUN_EVENTS = EXPORT_ROOT / f'RUN_EVENTS_{MULTIMODAL_RUN_NAME}.jsonl'
DISCONNECT_POINTS = EXPORT_ROOT / f'DISCONNECT_POINTS_{MULTIMODAL_RUN_NAME}.jsonl'
DONE_MARKER = EXPORT_ROOT / f'TRAINING_DONE_{MULTIMODAL_RUN_NAME}.json'
PHASE_DIR = EXPORT_ROOT / f'PHASES_{MULTIMODAL_RUN_NAME}'
PHASE_DIR.mkdir(parents=True, exist_ok=True)
FORCE_ALL = os.environ.get('FORCE_ALL', '0') == '1'
FORCE_PHASES = {x.strip() for x in os.environ.get('FORCE_PHASE', '').split(',') if x.strip()}
FORCE_MERGE = os.environ.get('FORCE_MERGE', '0') == '1'
FORCE_EXPORT = os.environ.get('FORCE_EXPORT', '0') == '1'
FORCE_BUNDLE = os.environ.get('FORCE_BUNDLE', '0') == '1'

def now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    tmp.replace(path)

def append_jsonl(path, payload):
    with Path(path).open('a', encoding='utf-8') as f:
        f.write(json.dumps(payload, ensure_ascii=False) + '\n')

def record_event(phase, event, **details):
    payload = {'ts': now_iso(), 'phase': phase, 'event': event, **details}
    append_jsonl(RUN_EVENTS, payload)
    print(json.dumps(payload, ensure_ascii=False))

def record_disconnect_point(phase, **details):
    append_jsonl(DISCONNECT_POINTS, {'ts': now_iso(), 'phase': phase, **details})

def phase_marker_path(name):
    safe = ''.join(ch if ch.isalnum() or ch in '._-' else '_' for ch in name)
    return PHASE_DIR / f'{safe}.json'

def phase_done(name):
    return False if FORCE_ALL or name in FORCE_PHASES else phase_marker_path(name).exists()

def mark_phase_done(name, artifacts=None, next_phase=None, **details):
    payload = {'phase': name, 'completed_at': now_iso(), 'artifacts': [str(x) for x in (artifacts or [])], 'next_phase': next_phase, **details}
    write_json(phase_marker_path(name), payload)
    write_json(RUN_STATE, {
        'version': 'v5_1_e2b_multimodal_export',
        'run_name': MULTIMODAL_RUN_NAME,
        'base_model': BASE_MODEL_ID,
        'adapter_dir': str(ADAPTER_DIR),
        'merged_multimodal_dir': str(MERGED_MM_DIR),
        'litertlm_final': str(LITERTLM_FINAL),
        'last_completed_phase': name,
        'updated_at': now_iso(),
    })
    record_event(name, 'phase_done', artifacts=payload['artifacts'], next_phase=next_phase)

if DONE_MARKER.exists() and not FORCE_ALL:
    print('Done marker exists:', DONE_MARKER)
record_disconnect_point('0_context_ready', repo=str(REPO_DIR), adapter_dir=str(ADAPTER_DIR))
mark_phase_done('0_context_ready', [RUN_STATE, RUN_EVENTS, DISCONNECT_POINTS], '1_install_deps')
print('HF token present:', bool(HF_TOKEN))
print('Adapter dir:', ADAPTER_DIR)
print('Export root:', EXPORT_ROOT)


## Phase 1 — Install dependencies

Installs current Transformers so `AutoModelForImageTextToText` can recognize `gemma4`. If Colab asks for a runtime restart after package upgrades, restart and rerun from Phase 0; markers will skip completed phases.


In [ ]:
# Phase 1: Install dependencies.
if phase_done('1_install_deps'):
    print('Skipping Phase 1; marker exists. If exporter fails, Phase 5/6 can still repair LiteRT and Transformers deps.')
else:
    cmds = [[sys.executable, '-m', 'pip', 'install', '-U', 'accelerate', 'peft', 'safetensors', 'sentencepiece', 'protobuf', 'packaging', 'hf_transfer']]
    cmds.append([sys.executable, '-m', 'pip', 'uninstall', '-y', 'Pillow', 'pillow', 'PIL'])
    cmds.append([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-cache-dir', 'Pillow==11.3.0'])
    if os.environ.get('SKIP_LITERT_INSTALL', '0') != '1':
        cmds.append([sys.executable, '-m', 'pip', 'install', '-U', '--pre', 'litert-torch-nightly', 'litert-lm', 'ai-edge-litert-nightly', 'litert-converter', 'sentencepiece', 'protobuf', 'huggingface_hub'])
    if os.environ.get('INSTALL_TORCH_NIGHTLY_FOR_LITERT', '1') == '1':
        cmds.append([sys.executable, '-m', 'pip', 'install', '-U', '--pre', 'torch', 'torchao', '--index-url', 'https://download.pytorch.org/whl/nightly/cu128'])
    elif os.environ.get('INSTALL_TORCHAO_FOR_PEFT', '1') == '1':
        cmds.append([sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', 'torchao>=0.16.0'])
    # Keep Transformers last. LiteRT dependency repair can otherwise leave the subprocess exporter on a release that does not know gemma4.
    if os.environ.get('SKIP_TRANSFORMERS_GIT', '0') == '1':
        cmds.append([sys.executable, '-m', 'pip', 'install', '-U', 'transformers'])
    else:
        cmds.append([sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', '--no-cache-dir', 'git+https://github.com/huggingface/transformers.git'])
    for cmd in cmds:
        record_event('1_install_deps', 'run', command=' '.join(cmd))
        subprocess.run(cmd, check=True)
    import importlib.metadata
    versions = {}
    for pkg in ['torch', 'torchao', 'transformers', 'peft', 'safetensors', 'Pillow', 'litert_torch', 'litert_lm']:
        try:
            versions[pkg] = importlib.metadata.version(pkg)
        except Exception as exc:
            versions[pkg] = f'unavailable: {exc}'
    write_json(EXPORT_ROOT / 'dependency_versions.json', versions)
    mark_phase_done('1_install_deps', [EXPORT_ROOT / 'dependency_versions.json'], '2_preflight')
print('Dependency phase complete. If torch was upgraded in this cell, restart runtime before heavy merge/export for the cleanest state.')


## Phase 2 — Base model and adapter preflight

Verifies the v5.1 adapter exists, `google/gemma-4-E2B-it` reports `model_type=gemma4`, and `AutoProcessor` is loadable before downloading full weights.


In [ ]:
# Phase 2: Base model / adapter preflight.
if phase_done('2_preflight'):
    print('Skipping Phase 2; marker exists.')
else:
    def ensure_pillow_importable():
        import importlib.util
        def clear_pil_modules():
            for name in list(sys.modules):
                if name == 'PIL' or name.startswith('PIL.'):
                    del sys.modules[name]
        def check_pillow():
            import PIL
            import PIL.Image  # noqa: F401
            import PIL.ImageDraw  # noqa: F401
            import PIL.ImageFont  # noqa: F401
            if importlib.util.find_spec('PIL.ImageText') is not None:
                import PIL.ImageText  # noqa: F401
            return PIL.__version__
        try:
            return check_pillow()
        except ImportError as exc:
            if '_Ink' not in str(exc) and 'PIL' not in str(exc):
                raise
            print('Repairing mixed Pillow install before importing transformers:', repr(exc))
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'Pillow', 'pillow', 'PIL'], check=False)
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-cache-dir', 'Pillow==11.3.0'], check=True)
            clear_pil_modules()
            return check_pillow()
    print('Pillow version:', ensure_pillow_importable())
    def clear_transformers_modules():
        for name in list(sys.modules):
            if name == 'transformers' or name.startswith('transformers.'):
                del sys.modules[name]
    def transformers_supports_gemma4():
        try:
            import transformers
            try:
                from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
                return 'gemma4' in CONFIG_MAPPING_NAMES, getattr(transformers, '__version__', 'unknown')
            except Exception:
                from transformers.models.auto.configuration_auto import CONFIG_MAPPING
                return 'gemma4' in CONFIG_MAPPING, getattr(transformers, '__version__', 'unknown')
        except Exception as exc:
            return False, f'import_failed: {exc}'
    supports_gemma4, transformers_version = transformers_supports_gemma4()
    if not supports_gemma4:
        print('Transformers does not recognize gemma4 yet:', transformers_version)
        print('Installing Transformers from GitHub main, then clearing imported transformers modules.')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', '--no-cache-dir', 'git+https://github.com/huggingface/transformers.git'], check=True)
        clear_transformers_modules()
        supports_gemma4, transformers_version = transformers_supports_gemma4()
    if not supports_gemma4:
        raise RuntimeError(f'Transformers still does not support gemma4 after repair: {transformers_version}. Restart runtime, rerun Phase 0, then rerun Phase 2.')
    print('Transformers version:', transformers_version, 'gemma4 support:', supports_gemma4)
    from transformers import AutoConfig, AutoProcessor
    adapter_config = ADAPTER_DIR / 'adapter_config.json'
    if not adapter_config.exists():
        raise FileNotFoundError(f'GemmaFit v5.1 LoRA adapter not found: {adapter_config}')
    config = AutoConfig.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=False)
    processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=False)
    model_type = getattr(config, 'model_type', '')
    if model_type != 'gemma4':
        raise RuntimeError(f'Expected gemma4 config, got model_type={model_type!r}. Update Transformers or BASE_MODEL_ID.')
    if 'E2B' not in BASE_MODEL_ID.upper():
        raise RuntimeError(f'This notebook expects an E2B base to match the v5.1 adapter. Got {BASE_MODEL_ID}')
    preflight = {
        'base_model_id': BASE_MODEL_ID,
        'model_type': model_type,
        'architectures': getattr(config, 'architectures', None),
        'adapter_dir': str(ADAPTER_DIR),
        'adapter_config_sha256': hashlib.sha256(adapter_config.read_bytes()).hexdigest(),
        'processor': {
            'class': processor.__class__.__name__,
            'has_image_processor': hasattr(processor, 'image_processor'),
            'has_feature_extractor': hasattr(processor, 'feature_extractor'),
            'has_tokenizer': hasattr(processor, 'tokenizer'),
        },
        'checked_at': now_iso(),
    }
    write_json(EXPORT_ROOT / 'multimodal_preflight.json', preflight)
    mark_phase_done('2_preflight', [EXPORT_ROOT / 'multimodal_preflight.json'], '3_merge_image_text_to_text')
print((EXPORT_ROOT / 'multimodal_preflight.json').read_text(encoding='utf-8'))


## Phase 3 — Merge v5.1 LoRA through `AutoModelForImageTextToText`

This is intentionally different from the previous text-only `AutoModelForCausalLM` merge. It preserves the multimodal config/processor files in the merged HF directory.


In [ ]:
# Phase 3: Merge LoRA adapter into ImageTextToText HF/safetensors export.
if phase_done('3_merge_image_text_to_text') and MERGED_MM_DIR.exists() and not FORCE_MERGE:
    print('Skipping Phase 3; merged dir exists:', MERGED_MM_DIR)
else:
    import gc, importlib, importlib.metadata, torch
    from packaging.version import Version
    from peft import PeftModel

    def clear_transformers_modules_local():
        for name in list(sys.modules):
            if name == 'transformers' or name.startswith('transformers.'):
                del sys.modules[name]

    def ensure_image_text_transformers():
        try:
            from transformers import AutoModelForImageTextToText, AutoProcessor
            from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
            if 'gemma4' not in CONFIG_MAPPING_NAMES:
                raise ImportError('transformers CONFIG_MAPPING_NAMES does not contain gemma4')
            return AutoModelForImageTextToText, AutoProcessor
        except Exception as exc:
            print('Repairing Transformers for Gemma 4 ImageTextToText path:', repr(exc))
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', '--no-cache-dir', 'git+https://github.com/huggingface/transformers.git'], check=True)
            clear_transformers_modules_local()
            from transformers import AutoModelForImageTextToText, AutoProcessor
            return AutoModelForImageTextToText, AutoProcessor

    def from_pretrained_with_dtype(model_cls, model_id_or_path, dtype_value, **kwargs):
        try:
            return model_cls.from_pretrained(model_id_or_path, dtype=dtype_value, **kwargs)
        except TypeError as exc:
            if 'dtype' not in str(exc):
                raise
            print('from_pretrained(dtype=...) unsupported; retrying with torch_dtype=...')
            return model_cls.from_pretrained(model_id_or_path, torch_dtype=dtype_value, **kwargs)

    AutoModelForImageTextToText, AutoProcessor = ensure_image_text_transformers()

    def ensure_torchao_for_peft_merge():
        minimum = Version('0.16.0')
        try:
            current = Version(importlib.metadata.version('torchao'))
        except importlib.metadata.PackageNotFoundError:
            current = None
        if current is None or current < minimum:
            print(f'torchao {current or "not installed"} is incompatible with current PEFT checks; installing torchao>=0.16.0')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', 'torchao>=0.16.0'], check=True)
            importlib.invalidate_caches()
            for name in list(sys.modules):
                if name == 'torchao' or name.startswith('torchao.'):
                    del sys.modules[name]

    def suppress_optional_torchao_in_peft():
        def _not_available(*args, **kwargs):
            return False
        for module in list(sys.modules.values()):
            if getattr(module, '__name__', '').startswith('peft') and hasattr(module, 'is_torchao_available'):
                setattr(module, 'is_torchao_available', _not_available)

    def load_adapter_for_merge(base_model):
        try:
            return PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
        except ImportError as exc:
            if 'torchao' not in str(exc).lower():
                raise
            print('PEFT torchao compatibility check failed; retrying with torchao marked unavailable.')
            suppress_optional_torchao_in_peft()
            return PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))

    ensure_torchao_for_peft_merge()
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    record_event('3_merge_image_text_to_text', 'load_base', base_model_id=BASE_MODEL_ID, dtype=str(dtype))
    base_model = from_pretrained_with_dtype(
        AutoModelForImageTextToText,
        BASE_MODEL_ID,
        dtype,
        token=HF_TOKEN,
        device_map='auto',
        trust_remote_code=False,
    )
    processor = AutoProcessor.from_pretrained(BASE_MODEL_ID, token=HF_TOKEN, trust_remote_code=False)
    merged = load_adapter_for_merge(base_model).merge_and_unload()
    MERGED_MM_DIR.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(str(MERGED_MM_DIR), safe_serialization=True, max_shard_size='4GB')
    processor.save_pretrained(str(MERGED_MM_DIR))
    manifest = {
        'base_model_id': BASE_MODEL_ID,
        'adapter_dir': str(ADAPTER_DIR),
        'merged_multimodal_dir': str(MERGED_MM_DIR),
        'model_loader': 'AutoModelForImageTextToText',
        'processor_loader': 'AutoProcessor',
        'dtype': str(dtype),
        'saved_at': now_iso(),
        'files': sorted(p.name for p in MERGED_MM_DIR.iterdir()),
    }
    write_json(EXPORT_ROOT / 'merged_multimodal_manifest.json', manifest)
    del merged, base_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    mark_phase_done('3_merge_image_text_to_text', [MERGED_MM_DIR, EXPORT_ROOT / 'merged_multimodal_manifest.json'], '4_hf_image_smoke')
print('Merged multimodal directory:', MERGED_MM_DIR)


## Phase 4 — HF image-text smoke before LiteRT export

Confirms the merged HF directory accepts an image via `AutoProcessor`. This does not prove the LiteRT artifact preserves the vision encoder; Pixel smoke is still required.


In [ ]:
# Phase 4: HF ImageTextToText smoke with a synthetic image.
if phase_done('4_hf_image_smoke'):
    print('Skipping Phase 4; marker exists.')
else:
    import gc, torch
    from PIL import Image, ImageDraw
    try:
        from transformers import AutoModelForImageTextToText, AutoProcessor
    except Exception as exc:
        print('Repairing Transformers before HF image smoke:', repr(exc))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', '--no-cache-dir', 'git+https://github.com/huggingface/transformers.git'], check=True)
        for name in list(sys.modules):
            if name == 'transformers' or name.startswith('transformers.'):
                del sys.modules[name]
        from transformers import AutoModelForImageTextToText, AutoProcessor

    def from_pretrained_with_dtype(model_cls, model_id_or_path, dtype_value, **kwargs):
        try:
            return model_cls.from_pretrained(model_id_or_path, dtype=dtype_value, **kwargs)
        except TypeError as exc:
            if 'dtype' not in str(exc):
                raise
            print('from_pretrained(dtype=...) unsupported; retrying with torch_dtype=...')
            return model_cls.from_pretrained(model_id_or_path, torch_dtype=dtype_value, **kwargs)

    smoke_dir = EXPORT_ROOT / 'hf_multimodal_smoke'
    smoke_dir.mkdir(parents=True, exist_ok=True)
    image_path = smoke_dir / 'chair_person_synthetic.png'
    img = Image.new('RGB', (512, 512), 'white')
    draw = ImageDraw.Draw(img)
    draw.rectangle([40, 330, 230, 360], outline='gray', width=8)
    draw.line([70, 360, 70, 470], fill='gray', width=8)
    draw.line([210, 360, 210, 470], fill='gray', width=8)
    draw.ellipse([310, 70, 370, 130], outline='black', width=6)
    draw.line([340, 130, 340, 260], fill='black', width=6)
    draw.line([340, 170, 260, 260], fill='black', width=6)
    draw.line([340, 260, 300, 400], fill='black', width=6)
    draw.line([340, 260, 390, 400], fill='black', width=6)
    img.save(image_path)
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
    model = from_pretrained_with_dtype(AutoModelForImageTextToText, str(MERGED_MM_DIR), dtype, device_map='auto', trust_remote_code=False)
    processor = AutoProcessor.from_pretrained(str(MERGED_MM_DIR), trust_remote_code=False)
    prompt = 'Return one JSON object with function=create_care_activity_log. Describe visible activity evidence only. No diagnosis.'
    image = Image.open(image_path)
    messages = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': prompt}]}]
    try:
        chat_text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        inputs = processor(text=chat_text, images=image, return_tensors='pt')
        smoke_input_mode = 'apply_chat_template_text_then_processor'
    except Exception as exc:
        print('apply_chat_template(text) failed; trying tokenize=True message with image object. Reason:', repr(exc))
        try:
            messages_with_image = [{'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': prompt}]}]
            inputs = processor.apply_chat_template(messages_with_image, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors='pt')
            smoke_input_mode = 'apply_chat_template_tokenize'
        except Exception as exc2:
            print('apply_chat_template(tokenize=True) failed; using direct processor call. Reason:', repr(exc2))
            inputs = processor(text=prompt, images=image, return_tensors='pt')
            smoke_input_mode = 'direct_processor_text_images'
    first_device = next(model.parameters()).device
    inputs = {k: (v.to(first_device) if hasattr(v, 'to') else v) for k, v in inputs.items()}
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=96, do_sample=False)
    try:
        raw = processor.batch_decode(generated, skip_special_tokens=True)[0]
    except Exception:
        raw = str(generated)
    write_json(smoke_dir / 'hf_image_text_smoke.json', {'image_path': str(image_path), 'input_mode': smoke_input_mode, 'prompt': prompt, 'raw_response': raw[-2000:], 'completed_at': now_iso()})
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    mark_phase_done('4_hf_image_smoke', [image_path, smoke_dir / 'hf_image_text_smoke.json'], '5_exporter_verify')
print((EXPORT_ROOT / 'hf_multimodal_smoke' / 'hf_image_text_smoke.json').read_text(encoding='utf-8'))


## Phase 5 — Verify LiteRT exporter and available vision/audio flags


In [ ]:
# Phase 5: Verify exporter CLI and available multimodal flags.
if phase_done('5_exporter_verify'):
    print('Skipping Phase 5; marker exists.')
else:
    help_path = EXPORT_ROOT / f'{ARTIFACT_PREFIX}-export_hf_help.txt'

    def run_exporter_help():
        proc = subprocess.run([sys.executable, '-m', 'litert_torch.generative.export_hf', '--help'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        text = proc.stdout or ''
        help_path.write_text(text, encoding='utf-8')
        return proc.returncode, text

    def help_is_valid(text):
        return (
            ('SYNOPSIS' in text or 'usage:' in text.lower() or 'POSITIONAL ARGUMENTS' in text)
            and '--prefill_lengths' in text
            and '--cache_length' in text
            and '--quantization_recipe' in text
        )

    def log_tail(text, chars=6000):
        tail = (text or '')[-chars:]
        print('----- export_hf help tail -----')
        print(tail)
        print('----- end export_hf help tail -----')

    def clear_litert_modules():
        prefixes = ('litert', 'litert_torch', 'ai_edge_litert', 'torchao', 'transformers')
        for name in list(sys.modules):
            if name in prefixes or name.startswith(prefixes):
                del sys.modules[name]

    def install_litert_exporter_deps():
        repair_cmds = [
            [sys.executable, '-m', 'pip', 'install', '-U', '--pre', 'litert-torch-nightly', 'litert-lm', 'ai-edge-litert-nightly', 'litert-converter', 'sentencepiece', 'protobuf', 'huggingface_hub'],
            [sys.executable, '-m', 'pip', 'install', '-U', '--pre', 'torch', 'torchao', '--index-url', 'https://download.pytorch.org/whl/nightly/cu128'],
            [sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', '--no-cache-dir', 'git+https://github.com/huggingface/transformers.git'],
        ]
        for cmd in repair_cmds:
            record_event('5_exporter_verify', 'repair_run', command=' '.join(cmd))
            subprocess.run(cmd, check=True)
        clear_litert_modules()

    rc, help_text = run_exporter_help()
    if rc != 0 and help_is_valid(help_text):
        print('Exporter help returned non-zero but contains valid Fire/argparse help; continuing.')
    elif rc != 0:
        print('Exporter help failed on first attempt. Log path:', help_path)
        log_tail(help_text)
        if os.environ.get('EXPORTER_AUTO_REPAIR', '1') == '1':
            print('Repairing LiteRT exporter dependencies once, then retrying export_hf --help.')
            install_litert_exporter_deps()
            rc, help_text = run_exporter_help()
            if rc != 0 and help_is_valid(help_text):
                print('Exporter help returned non-zero after repair but contains valid help; continuing.')
            elif rc != 0:
                log_tail(help_text)
                raise RuntimeError('LiteRT exporter still fails after dependency repair. Restart Colab runtime, rerun Phase 0, then rerun Phase 5. Log: ' + str(help_path))
        else:
            raise RuntimeError('LiteRT exporter failed and EXPORTER_AUTO_REPAIR=0. Log: ' + str(help_path))

    candidate_flags = [
        '--task', '--export_vision_encoder', '--vision_encoder_quantization_recipe', '--bundle_litert_lm', '--auto_model_override',
        '--include_vision', '--export_vision', '--enable_vision', '--with_vision',
        '--include_audio', '--export_audio', '--enable_audio', '--with_audio',
        '--max_num_images', '--max_images', '--image_seq_length', '--audio_seq_length',
    ]
    available = {flag: (flag in help_text) for flag in candidate_flags}
    missing_required = [flag for flag in ['--prefill_lengths', '--cache_length', '--quantization_recipe'] if flag not in help_text]
    if missing_required:
        log_tail(help_text)
        raise RuntimeError(f'Exporter is missing required flags: {missing_required}. See {help_path}')
    if not available.get('--task'):
        raise RuntimeError('Exporter help is valid but does not expose --task; cannot request image_text_to_text export safely. Log: ' + str(help_path))
    write_json(EXPORT_ROOT / 'exporter_multimodal_capability.json', {'help_path': str(help_path), 'available_multimodal_flags': available, 'help_returncode': rc, 'checked_at': now_iso()})
    mark_phase_done('5_exporter_verify', [help_path, EXPORT_ROOT / 'exporter_multimodal_capability.json'], '6_litert_export_matrix')
print((EXPORT_ROOT / 'exporter_multimodal_capability.json').read_text(encoding='utf-8'))


## Phase 6 — LiteRT-LM export matrix

The first successful variant is copied to the final Drive artifact. Export success does not guarantee image support; Pixel `litert_image_prompt_infer` is the final authority.


In [ ]:
# Phase 6: Export LiteRT-LM matrix from merged multimodal HF directory.
if phase_done('6_litert_export_matrix') and LITERTLM_FINAL.exists() and not FORCE_EXPORT:
    print('Skipping Phase 6; final artifact exists:', LITERTLM_FINAL)
else:
    capability = json.loads((EXPORT_ROOT / 'exporter_multimodal_capability.json').read_text(encoding='utf-8'))
    available = capability.get('available_multimodal_flags', {})



    def subprocess_export_env_status():
        script = """
import importlib.util
import json
status = {}
try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    status['transformers_version'] = getattr(transformers, '__version__', 'unknown')
    status['gemma4'] = 'gemma4' in CONFIG_MAPPING_NAMES
except Exception as exc:
    status['transformers_error'] = repr(exc)
    status['gemma4'] = False
try:
    import torch
    status['torch_version'] = getattr(torch, '__version__', 'unknown')
except Exception as exc:
    status['torch_error'] = repr(exc)
try:
    import torchao
    status['torchao_version'] = getattr(torchao, '__version__', 'unknown')
    try:
        status['torchao_pt2e'] = importlib.util.find_spec('torchao.quantization.pt2e') is not None
    except Exception:
        status['torchao_pt2e'] = False
    try:
        status['torchao_quantize_pt2e'] = importlib.util.find_spec('torchao.quantization.pt2e.quantize_pt2e') is not None
    except Exception:
        status['torchao_quantize_pt2e'] = False
except Exception as exc:
    status['torchao_error'] = repr(exc)
    status['torchao_pt2e'] = False
    status['torchao_quantize_pt2e'] = False
try:
    import litert_converter  # noqa: F401
    status['litert_converter'] = True
except Exception as exc:
    status['litert_converter'] = False
    status['litert_converter_error'] = repr(exc)
try:
    import litert_torch.generative.export_hf  # noqa: F401
    status['litert_torch_export_hf'] = True
except Exception as exc:
    status['litert_torch_export_hf'] = False
    status['litert_torch_error'] = repr(exc)
print(json.dumps(status))
"""
        proc = subprocess.run([sys.executable, '-c', script], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        text = (proc.stdout or '').strip().splitlines()[-1] if (proc.stdout or '').strip() else '{}'
        try:
            return json.loads(text)
        except Exception:
            return {'gemma4': False, 'litert_converter': False, 'torchao_pt2e': False, 'litert_torch_export_hf': False, 'raw': proc.stdout, 'returncode': proc.returncode}

    def install_full_export_stack_for_subprocess():
        repair_cmds = [
            [sys.executable, '-m', 'pip', 'install', '-U', '--pre', 'litert-torch-nightly', 'litert-lm', 'ai-edge-litert-nightly', 'litert-converter', 'sentencepiece', 'protobuf', 'huggingface_hub'],
            [sys.executable, '-m', 'pip', 'install', '-U', '--pre', 'torch', 'torchao', '--index-url', 'https://download.pytorch.org/whl/nightly/cu128'],
            [sys.executable, '-m', 'pip', 'install', '-U', '--no-deps', '--no-cache-dir', 'git+https://github.com/huggingface/transformers.git'],
        ]
        for cmd in repair_cmds:
            record_event('6_litert_export_matrix', 'subprocess_stack_repair_run', command=' '.join(cmd))
            subprocess.run(cmd, check=True)

    def ensure_export_subprocess_ready():
        status = subprocess_export_env_status()
        print('Exporter subprocess env status:', status)
        ready = (
            status.get('gemma4')
            and status.get('litert_converter')
            and status.get('torchao_pt2e')
            and status.get('litert_torch_export_hf')
        )
        if not ready:
            print('Repairing full LiteRT export stack once: LiteRT packages + PyTorch/torchao nightly + Gemma4 Transformers.')
            install_full_export_stack_for_subprocess()
            status = subprocess_export_env_status()
        print('Exporter subprocess env status after repair:', status)
        ready = (
            status.get('gemma4')
            and status.get('litert_converter')
            and status.get('torchao_pt2e')
            and status.get('litert_torch_export_hf')
        )
        if not ready:
            raise RuntimeError('Exporter subprocess is not ready after full stack repair. Restart Colab runtime, rerun Phase 0, Phase 5, then Phase 6. Status: ' + json.dumps(status))
        return status

    ensure_export_subprocess_ready()

    def optional_export_flags():
        flags = []
        if available.get('--task'):
            flags.append('--task=image_text_to_text')
        if available.get('--export_vision_encoder'):
            flags.append('--export_vision_encoder=True')
        else:
            for flag in ['--include_vision', '--export_vision', '--enable_vision', '--with_vision']:
                if available.get(flag):
                    flags.append(flag)
                    break
        if available.get('--vision_encoder_quantization_recipe'):
            flags.append('--vision_encoder_quantization_recipe=dynamic_wi8_afp32')
        if available.get('--bundle_litert_lm'):
            flags.append('--bundle_litert_lm=True')
        for flag in ['--include_audio', '--export_audio', '--enable_audio', '--with_audio']:
            if available.get(flag):
                flags.append(flag)
                break
        if available.get('--max_num_images'):
            flags += ['--max_num_images', '1']
        elif available.get('--max_images'):
            flags += ['--max_images', '1']
        return flags

    def run_streaming(command, log_path):
        log_path.parent.mkdir(parents=True, exist_ok=True)
        with log_path.open('w', encoding='utf-8', errors='replace') as log_handle:
            log_handle.write('COMMAND: ' + ' '.join(command) + '\n')
            log_handle.flush()
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end='')
                log_handle.write(line)
            return process.wait()

    export_results, selected = [], None
    for variant in EXPORT_VARIANTS:
        variant_name = variant['name']
        local_dir = LOCAL_WORK_ROOT / variant_name
        drive_variant_dir = EXPORT_ROOT / variant_name
        log_path = EXPORT_ROOT / f'{ARTIFACT_PREFIX}-{variant_name}-export.log'
        if local_dir.exists():
            shutil.rmtree(local_dir)
        local_dir.mkdir(parents=True, exist_ok=True)
        drive_variant_dir.mkdir(parents=True, exist_ok=True)
        cmd = [
            sys.executable, '-m', 'litert_torch.generative.export_hf',
            str(MERGED_MM_DIR), str(local_dir),
            f"--prefill_lengths={variant['prefill_lengths']}",
            f"--cache_length={variant['cache_length']}",
            '--quantization_recipe=dynamic_wi8_afp32',
            '--externalize_embedder=True',
        ] + optional_export_flags()
        record_event('6_litert_export_matrix', 'export_start', variant=variant_name, command=' '.join(cmd))
        start = time.time()
        rc = run_streaming(cmd, log_path)
        litert_files = sorted(local_dir.rglob('*.litertlm'))
        success = rc == 0 and bool(litert_files)
        result = {'variant': variant, 'returncode': rc, 'elapsed_sec': round(time.time() - start, 1), 'log_path': str(log_path), 'success': success, 'litert_files': [str(p) for p in litert_files], 'optional_flags': optional_export_flags()}
        if success:
            src_litert = litert_files[0]
            variant_final = drive_variant_dir / f'{ARTIFACT_PREFIX}-{variant_name}.litertlm'
            shutil.copy2(src_litert, variant_final)
            result['drive_litertlm'] = str(variant_final)
            if selected is None:
                shutil.copy2(src_litert, LITERTLM_FINAL)
                selected = {'variant': variant_name, 'source': str(src_litert), 'final': str(LITERTLM_FINAL), 'size_bytes': LITERTLM_FINAL.stat().st_size}
        else:
            record_event('6_litert_export_matrix', 'export_failed', variant=variant_name, log_path=str(log_path), returncode=rc)
        export_results.append(result)
        write_json(EXPORT_ROOT / 'litert_multimodal_export_matrix_partial.json', {'results': export_results, 'selected': selected})
    matrix = {'base_model_id': BASE_MODEL_ID, 'adapter_dir': str(ADAPTER_DIR), 'merged_multimodal_dir': str(MERGED_MM_DIR), 'results': export_results, 'selected': selected, 'completed_at': now_iso()}
    write_json(EXPORT_ROOT / 'litert_multimodal_export_matrix.json', matrix)
    if selected is None:
        raise RuntimeError('No LiteRT-LM export variant succeeded. Inspect logs under ' + str(EXPORT_ROOT))
    mark_phase_done('6_litert_export_matrix', [LITERTLM_FINAL, EXPORT_ROOT / 'litert_multimodal_export_matrix.json'], '7_package_android_artifacts')
print((EXPORT_ROOT / 'litert_multimodal_export_matrix.json').read_text(encoding='utf-8')[:4000])


## Phase 7 — Package artifacts and Pixel smoke commands


In [ ]:
# Phase 7: Package local artifacts and Pixel smoke commands.
if phase_done('7_package_android_artifacts') and BUNDLE_ZIP.exists() and not FORCE_BUNDLE:
    print('Skipping Phase 7; bundle exists:', BUNDLE_ZIP)
else:
    if not LITERTLM_FINAL.exists():
        raise FileNotFoundError(f'Final LiteRT-LM artifact missing: {LITERTLM_FINAL}')
    bundle_root = LOCAL_WORK_ROOT / 'android_bundle'
    if bundle_root.exists():
        shutil.rmtree(bundle_root)
    models_dir = bundle_root / 'models'
    metrics_dir = bundle_root / 'finetune' / 'metrics'
    models_dir.mkdir(parents=True, exist_ok=True)
    metrics_dir.mkdir(parents=True, exist_ok=True)
    mm_name = f'{ARTIFACT_PREFIX}.litertlm'
    app_name = 'gemmafit-v5-e2b-evidence-router.litertlm'
    shutil.copy2(LITERTLM_FINAL, models_dir / mm_name)
    shutil.copy2(LITERTLM_FINAL, models_dir / app_name)
    smoke_image = EXPORT_ROOT / 'hf_multimodal_smoke' / 'chair_person_synthetic.png'
    if smoke_image.exists():
        shutil.copy2(smoke_image, bundle_root / 'debug_phone_current.png')
    training_done = {
        'version': 'v5_1_e2b_multimodal_export',
        'status': 'conversion_complete_pixel_unverified',
        'android_ready': False,
        'base_model_id': BASE_MODEL_ID,
        'adapter_dir': str(ADAPTER_DIR),
        'merged_multimodal_dir': str(MERGED_MM_DIR),
        'litertlm_path': f'models/{mm_name}',
        'app_resolver_filename': app_name,
        'required_pixel_smoke': 'content://com.gemmafit.debug/litert_image_prompt_infer?image=debug_phone_current.png',
        'boundary': 'Pixel image smoke must pass before Android-ready labeling. TF_LITE_VISION_ENCODER missing means text-only graph.',
        'created_at': now_iso(),
    }
    write_json(metrics_dir / 'training_done_v5_e2b_multimodal.json', training_done)
    smoke_lines = [
        '# Pixel smoke commands for GemmaFit multimodal LiteRT-LM export',
        '$ErrorActionPreference = "Stop"',
        'adb devices',
        'adb shell mkdir -p /sdcard/Android/data/com.gemmafit/files',
        f'adb push models\\{mm_name} /sdcard/Android/data/com.gemmafit/files/{app_name}',
        'adb push debug_phone_current.png /sdcard/Android/data/com.gemmafit/files/debug_phone_current.png',
        'adb shell content read --uri content://com.gemmafit.debug/model_readiness',
        'adb shell content read --uri "content://com.gemmafit.debug/litert_image_prompt_infer?image=debug_phone_current.png"',
        '',
        '# PASS: success=true and no TF_LITE_VISION_ENCODER missing error.',
        '# FAIL: TF_LITE_VISION_ENCODER not found means this export is still text-only.',
    ]
    (bundle_root / 'PIXEL_SMOKE_COMMANDS.ps1').write_text('\n'.join(smoke_lines) + '\n', encoding='utf-8')
    (bundle_root / 'README_PIXEL_SMOKE.md').write_text('# GemmaFit v5.1 E2B multimodal Pixel smoke\n\nInstall the debug APK first, then run PIXEL_SMOKE_COMMANDS.ps1. Do not mark Android-ready until litert_image_prompt_infer passes.\n', encoding='utf-8')
    if BUNDLE_ZIP.exists():
        BUNDLE_ZIP.unlink()
    shutil.make_archive(str(BUNDLE_ZIP.with_suffix('')), 'zip', bundle_root)
    mark_phase_done('7_package_android_artifacts', [BUNDLE_ZIP, metrics_dir / 'training_done_v5_e2b_multimodal.json'], '8_done_marker')
print('Bundle:', BUNDLE_ZIP)
print('Bundle exists:', BUNDLE_ZIP.exists(), 'size:', BUNDLE_ZIP.stat().st_size if BUNDLE_ZIP.exists() else None)


## Phase 8 — Done marker


In [ ]:
# Phase 8: Write final done marker for Colab export.
if phase_done('8_done_marker') and DONE_MARKER.exists():
    print('Skipping Phase 8; done marker exists:', DONE_MARKER)
else:
    matrix_path = EXPORT_ROOT / 'litert_multimodal_export_matrix.json'
    matrix = json.loads(matrix_path.read_text(encoding='utf-8')) if matrix_path.exists() else {}
    done = {
        'version': 'v5_1_e2b_multimodal_export',
        'run_name': MULTIMODAL_RUN_NAME,
        'run_suffix': MULTIMODAL_RUN_SUFFIX,
        'finished_at': now_iso(),
        'status': 'conversion_complete_pixel_unverified',
        'android_ready': False,
        'base_model_id': BASE_MODEL_ID,
        'source_run_name': SOURCE_RUN_NAME,
        'adapter_path': str(ADAPTER_DIR),
        'merged_multimodal_path': str(MERGED_MM_DIR),
        'litertlm_path': str(LITERTLM_FINAL),
        'bundle_zip': str(BUNDLE_ZIP),
        'selected_export': matrix.get('selected'),
        'pixel_smoke_route': 'content://com.gemmafit.debug/litert_image_prompt_infer?image=debug_phone_current.png',
        'run_state': str(RUN_STATE),
        'run_events': str(RUN_EVENTS),
        'disconnect_points': str(DISCONNECT_POINTS),
        'notes': ['No retraining.', 'Merge uses AutoModelForImageTextToText and AutoProcessor.', 'Pixel image smoke is required before Android-ready labeling.'],
    }
    write_json(DONE_MARKER, done)
    mark_phase_done('8_done_marker', [DONE_MARKER], None)
print(DONE_MARKER.read_text(encoding='utf-8'))


## Expected outcomes

- If HF image smoke passes but Pixel `litert_image_prompt_infer` fails with `TF_LITE_VISION_ENCODER not found in the model`, the adapter merge is usable but this exporter path still emitted a text-only graph.
- If the 4096-cache export OOMs, rerun with `FORCE_EXPORT=1` and let the `ctx2048/prefill512` variants complete.
- If Pixel image smoke passes, download the bundle and keep the app-facing filename `gemmafit-v5-e2b-evidence-router.litertlm` unless `CoachModelResolver` is updated to prefer a `-mm` filename.
